In [1]:
from __future__ import annotations

import os
import numpy as np
import h5py

import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
class ROIGridReviewer:

    PAGE_SIZE = 25
    ROWS = 5
    COLS = 5

    def __init__(self, session_path):

        self.session_path = session_path

        self.load_session()
        self.load_labels()

        self.current_page = 0
        self.selected_index = 0

        self.sort_rois()

        self.build_ui()

        self.draw_grid()
        self.draw_detail()
        self.update_status()
        self.update_grid_buttons()


    # -----------------------------------------------------
    # Load imaging data
    # -----------------------------------------------------

    def load_session(self):

        ops_path = os.path.join(
            self.session_path,
            "suite2p",
            "plane0",
            "ops.npy"
        )

        mask_path = os.path.join(
            self.session_path,
            "qc_results",
            "masks.npy"
        )


        ops = np.load(
            ops_path,
            allow_pickle=True
        ).item()


        self.mean_img = ops["meanImg"]


        self.masks = np.load(
            mask_path
        )

        fluo = np.load(
            os.path.join(
                self.session_path,
                "qc_results",
                "fluo.npy"
            )
        )
        
        neuropil = np.load(
            os.path.join(
                self.session_path,
                "qc_results",
                "neuropil.npy"
            )
        )
        
        neucoeff = float(
            ops.get("neucoeff", 0.7)
        )
        
        sig_baseline = int(
            ops.get("sig_baseline", 10)
        )
        
        dff = fluo.astype(np.float32) - neucoeff * neuropil.astype(np.float32)
        
        from scipy.ndimage import gaussian_filter
        
        f0 = gaussian_filter(
            dff,
            [0, sig_baseline]
        )
        
        self.dff = (dff - f0) / (f0 + 1e-10)


        self.n_rois = int(
            self.masks.max()
        )

        self.rois = list(
            range(self.n_rois)
        )


    # -----------------------------------------------------
    # Load ROI_label.h5
    # -----------------------------------------------------

    def load_labels(self):

        label_path = os.path.join(
            self.session_path,
            "ROI_label.h5"
        )


        with h5py.File(label_path, "r") as f:

            self.good_roi = set(
                f["good_roi"][:].astype(int)
            )

            self.bad_roi = set(
                f["bad_roi"][:].astype(int)
            )


    # -----------------------------------------------------
    # Save labels
    # -----------------------------------------------------

    def save_labels(self):

        label_path = os.path.join(
            self.session_path,
            "ROI_label.h5"
        )


        with h5py.File(label_path, "w") as f:

            f.create_dataset(
                "good_roi",
                data=np.array(
                    sorted(self.good_roi),
                    dtype=np.int32
                )
            )

            f.create_dataset(
                "bad_roi",
                data=np.array(
                    sorted(self.bad_roi),
                    dtype=np.int32
                )
            )


        print("Saved ROI_label.h5")


    # -----------------------------------------------------
    # Sort: labeled first
    # -----------------------------------------------------

    def sort_rois(self):

        def priority(roi):
    
            if roi in self.good_roi:
                return 0      # Good first
    
            if roi in self.bad_roi:
                return 1      # Then bad
    
            return 2          # Then unlabeled
    
        self.rois = sorted(self.rois, key=priority)



    def update_grid_buttons(self):

        rois = self.page_rois()
    
        for idx, button in enumerate(self.buttons):
    
            if idx < len(rois):
    
                roi = rois[idx]
    
                button.description = f"{roi}"
    
                if roi in self.good_roi:
                    button.button_style = "success"
    
                elif roi in self.bad_roi:
                    button.button_style = "danger"
    
                else:
                    button.button_style = ""
    
                # Highlight currently selected ROI
                if idx == self.selected_index:
                    button.layout.border = "3px solid black"
                else:
                    button.layout.border = "1px solid lightgray"
    
            else:
                button.description = ""
                button.button_style = ""
                button.layout.border = ""


    # -----------------------------------------------------
    # UI
    # -----------------------------------------------------

    def build_ui(self):

        self.grid_output = widgets.Output()

        self.detail_output = widgets.Output()


        self.status = widgets.HTML()


        self.good_button = widgets.Button(
            description="✓ Good",
            button_style="success"
        )


        self.bad_button = widgets.Button(
            description="✗ Bad",
            button_style="danger"
        )


        self.skip_button = widgets.Button(
            description="→ Skip"
        )


        self.prev_button = widgets.Button(
            description="◀ Prev"
        )


        self.next_button = widgets.Button(
            description="Next ▶"
        )


        self.buttons = []

        for i in range(self.PAGE_SIZE):

            b = widgets.Button(
                description=f"{i}",
                layout=widgets.Layout(
                    width="70px",
                    height="30px"
                )
            )

            b.idx = i

            b.on_click(
                self.select_roi
            )

            self.buttons.append(b)


        self.good_button.on_click(
            lambda x: self.label_roi(1)
        )

        self.bad_button.on_click(
            lambda x: self.label_roi(0)
        )

        self.skip_button.on_click(
            lambda x: self.next_roi()
        )

        self.prev_button.on_click(
            lambda x: self.prev_page()
        )

        self.next_button.on_click(
            lambda x: self.next_page()
        )


        grid_buttons = widgets.GridBox(
            self.buttons,
            layout=widgets.Layout(
                grid_template_columns=
                "repeat(5, 75px)"
            )
        )


        controls = widgets.HBox(
            [
                self.prev_button,
                self.good_button,
                self.bad_button,
                self.skip_button,
                self.next_button
            ]
        )


        display(self.status)

        display(
            widgets.HBox(
                [
                    widgets.VBox(
                        [
                            self.grid_output,
                            grid_buttons
                        ]
                    ),
                    self.detail_output
                ]
            )
        )


        display(
            controls
        )


    # -----------------------------------------------------
    # Current page ROIs
    # -----------------------------------------------------

    def page_rois(self):

        start = (
            self.current_page *
            self.PAGE_SIZE
        )

        return self.rois[
            start:start+self.PAGE_SIZE
        ]


    # -----------------------------------------------------
    # Draw grid
    # -----------------------------------------------------

    def draw_grid(self):

        rois = self.page_rois()
    
        with self.grid_output:
    
            clear_output(wait=True)
    
            fig, axes = plt.subplots(
                self.ROWS,
                self.COLS,
                figsize=(10, 10)
            )
    
            axes = axes.flat
    
            for idx, ax in enumerate(axes):
    
                if idx >= len(rois):
                    ax.axis("off")
                    continue
    
                roi = rois[idx]
    
                ax.imshow(
                    self.mean_img,
                    cmap="gray"
                )
    
                mask = self.masks == (roi + 1)
    
                ax.contour(
                    mask,
                    colors="orange",
                    linewidths=1
                )
    
    
                # Label color
                if roi in self.good_roi:
                    color = "green"
    
                elif roi in self.bad_roi:
                    color = "red"
    
                else:
                    color = "gray"
    
    
                # Thick border
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_color(color)
                    spine.set_linewidth(4)
    
    
                ax.set_title(
                    f"ROI {roi}",
                    fontsize=10,
                    color=color
                )
    
                ax.set_xticks([])
                ax.set_yticks([])
    
    
            plt.tight_layout()
            plt.show()



    # -----------------------------------------------------
    # Detail image
    # -----------------------------------------------------

    def draw_detail(self):

        roi = self.page_rois()[self.selected_index]
    
        with self.detail_output:
    
            clear_output(wait=True)
    
            fig, (ax_img, ax_trace) = plt.subplots(
                2,
                1,
                figsize=(5,8)
            )
    
            # ----------------------------
            # ROI image
            # ----------------------------
    
            ax_img.imshow(
                self.mean_img,
                cmap="gray"
            )
    
            mask = self.masks == (roi + 1)
    
            ax_img.contour(
                mask,
                colors="orange",
                linewidths=2
            )
    
            ax_img.set_title(f"ROI {roi}")
            ax_img.axis("off")
    
            # ----------------------------
            # Fluorescence trace
            # ----------------------------
    
            ax_trace.plot(
                self.dff[roi],
                linewidth=0.8
            )
    
            ax_trace.set_title("ΔF/F Trace")
    
            ax_trace.set_xlabel("Frame")
    
            ax_trace.set_ylabel("ΔF/F")
    
            ax_trace.spines["top"].set_visible(False)
            ax_trace.spines["right"].set_visible(False)
    
            plt.tight_layout()
    
            plt.show()


    # -----------------------------------------------------
    # Button actions
    # -----------------------------------------------------

    def select_roi(self, button):

        if button.idx < len(
            self.page_rois()
        ):

            self.selected_index = (
                button.idx
            )

            self.update_grid_buttons()

            self.draw_detail()


    def label_roi(self, value):

        roi = self.page_rois()[self.selected_index]
    
        if value == 1:
    
            self.good_roi.add(roi)
            self.bad_roi.discard(roi)
    
        elif value == 0:
    
            self.bad_roi.add(roi)
            self.good_roi.discard(roi)
    
        # Save immediately
        self.save_labels()
    
        # -----------------------------
        # Automatically advance
        # -----------------------------
        if self.selected_index < len(self.page_rois()) - 1:
    
            # Next ROI on current page
            self.selected_index += 1
    
        elif (self.current_page + 1) * self.PAGE_SIZE < len(self.rois):
    
            # First ROI on next page
            self.current_page += 1
            self.selected_index = 0
    
        # Refresh everything
        self.draw_grid()
        self.update_grid_buttons()
        self.draw_detail()
        self.update_status()



    def next_roi(self):

        if self.selected_index < len(
            self.page_rois()
        )-1:

            self.selected_index += 1

        self.update_grid_buttons()
        self.draw_detail()



    def next_page(self):

        if (
            (self.current_page+1)
            *
            self.PAGE_SIZE
            <
            len(self.rois)
        ):

            self.current_page += 1
            self.selected_index = 0

            self.draw_grid()
            self.draw_detail()
            self.update_grid_buttons()



    def prev_page(self):

        if self.current_page > 0:

            self.current_page -= 1
            self.selected_index = 0

            self.draw_grid()
            self.draw_detail()
            self.update_grid_buttons()



    def update_status(self):
        total_pages = (
            len(self.rois) + self.PAGE_SIZE - 1
        ) // self.PAGE_SIZE

        self.status.value = (
            f"""
            <b>{os.path.basename(self.session_path)}</b>
            |
            Page {self.current_page + 1} of {total_pages}
            |
            Good: {len(self.good_roi)}
            |
            Bad: {len(self.bad_roi)}
            |
            Unlabeled:
            {self.n_rois - len(self.good_roi) - len(self.bad_roi)}
            """
        )

In [3]:
session = "/storage/project/r-fnajafi3-0/shared/2P_Imaging/SA18_LG/SA18_20260220"

In [4]:
reviewer = ROIGridReviewer(session)

HTML(value='')